# 02 — Raw race data EDA

**Goal:** understand the raw race laps well enough to (1) write the spec for the processed layer and (2) see the degradation signal that the Stage 1 model will learn.

**Data:** 92 races (2022–2025), 101,290 race laps, from `data/raw` (see `docs/DECISIONS.md` D2).

**How to read this notebook:** a column guide, then a one-page summary, then one section per question. Each section ends with a **Finding** box: what we saw and what we decided.


## Column guide

### `laps`: one row = one lap of one driver in one race

| Column | Meaning |
|---|---|
| `Season`, `Round`, `Session` | Which race. `Round` is only unique *within* a season. `Session` is always `R` here |
| `Driver`, `DriverNumber`, `Team` | Who drove the lap |
| `LapNumber` | Lap of the race (1 = first lap) |
| `LapTime` | **Duration**: how long the lap took |
| `Sector1Time` … `Sector3Time` | **Duration** of each of the three track sectors |
| `Time` | **Session clock**: when the driver crossed the line to finish this lap |
| `LapStartTime` | **Session clock**: when the lap started |
| `PitInTime` / `PitOutTime` | **Session clock**: when the car entered / left the pit lane. Only filled on the in-lap / out-lap |
| `Sector1SessionTime` … | **Session clock**: when each sector ended |
| `LapStartDate` | Real date and time of the lap start. **Always empty** (needs telemetry, which we don't load) |
| `Stint` | Tyre-set counter: 1 = first set, +1 after each pit stop |
| `Compound` | Tyre type: SOFT, MEDIUM, HARD (dry); INTERMEDIATE, WET (rain) |
| `TyreLife` | Age of the current tyre set in laps (resets at each stop; used sets start older) |
| `FreshTyre` | True if the set was new when fitted, False if already used (e.g. in qualifying) |
| `TrackStatus` | Every track status during the lap, as a string of digits in order (1 clear, 2 yellow, 4 safety car, 5 red flag, 6 VSC, 7 VSC ending) |
| `Position` | Race position at the end of the lap |
| `SpeedI1`, `SpeedI2`, `SpeedFL`, `SpeedST` | Speed-trap readings (km/h) at fixed points on the track |
| `IsPersonalBest` | Driver's fastest lap of the race so far |
| `Deleted`, `DeletedReason` | Lap time cancelled by race control, almost always for track limits |
| `IsAccurate` | FastF1's own check that the lap timing is reliable |
| `FastF1Generated` | Row filled in by FastF1 rather than coming from the timing feed |
| `event_name`, `location`, `event_format`, `session_start_utc` | Added from our metadata: race name, circuit, weekend format, race start time |
| `*_sec` | Added in this notebook: the time columns above converted to seconds |

**Three kinds of time:** a *duration* (`LapTime`: how long), a *session-clock* reading (`Time`: when, counted from a stopwatch that starts ~1 h before the race), and a *real datetime* (`session_start_utc`). Raw durations and clock readings are stored as integer **nanoseconds**.

### `weather`: one row = one reading, about every 60 s
`Time` (session clock), `AirTemp`, `TrackTemp` (°C), `Humidity` (%), `Pressure` (mbar), `Rainfall` (True/False), `WindDirection` (°), `WindSpeed` (m/s).

### `track_status`: one row = one change of track status
`Time` (session clock), `Status` (single digit), `Message` (e.g. `SCDeployed`). This is the code book for `laps.TrackStatus`.

### Not used yet
`results` (grid, finishing position, status per driver) and `race_control` (flags, penalties, messages). Needed later for the simulator and race-winner model.


## Summary

| Step | What we found | What we decided |
|---|---|---|
| **Q1 Rows** | One row per driver per lap; `(Season, Round, Driver, LapNumber)` has 0 duplicates. Retirements show as fewer laps | That is the processed table's key. Always group by `Season` + `Round` |
| **Q2 Missing** | 1.5% of laps have no lap time, mostly safety-car or red-flag laps, in a handful of races | Flag `has_lap_time`; those laps are excluded anyway |
| **Q3 Track status** | A lap's status lists everything that happened during it. Green ≈ 89.7 s avg, safety car ≈ 123 s | Turn into yes/no flags: yellow, safety car, VSC, red flag |
| **Q4 Pit laps** | In-lap +2.5 s, out-lap +18 s: that's pit-stop time, not tyre wear. The laps *around* a stop show the tyre gap (99.5 → 96.9 s) | Flag in/out laps; exclude from degradation; reuse for pit-loss later |
| **Q5 Outliers** | 13.6% of laps "inaccurate" (mostly pit, lap 1, non-green laps); 1.3% deleted for track limits | Keep both as flags |
| **Q6 Tyres** | `Stint` counts sets; `TyreLife` resets at stops; used sets start older. `"None"`/`"nan"` compounds are text, not missing. Median stint 19 laps | Fix compound text; wet flag; skip stints < 5 laps for slopes |
| **Q7 Weather** | Reading every 60 s. Track temp changes a little within a race, a lot between races (17–54 °C). Rain sensor misses wet tracks (Imola 2022: 0 rain readings, 329 wet-tyre laps) | Attach latest reading before each lap; temperature is a feature; mark wet laps by **tyre**, not rain sensor |
| **Q8 Fuel** | Cars get ~0.06 s/lap faster as fuel burns (0.03 Monaco – 0.08 Spa). Within a stint this hides tyre wear | Fuel-correct lap times before measuring degradation; per-track value in config |
| **Q9 Traffic** | Within ~1 s of the car ahead, laps are a few tenths slower (dirty air). Raw comparison overstates it (early-race laps are close *and* heavy) | Store `gap_ahead_sec`; flag `in_traffic` (< 1 s) and exclude for degradation |
| **Q10 Degradation** | After all corrections, lap time rises almost linearly with tyre age. Bahrain 2022 median slope: SOFT 0.18, MEDIUM 0.15, HARD 0.13 s/lap | Linear-in-tyre-age base model is a good start. The fuel value shifts every slope one-for-one, so it must be estimated well |

**The recipe for a degradation curve** (= the Stage 1 pipeline):
clean laps → remove traffic → fuel-correct → compare each stint with its own start → slope per stint.

**Still open:** why some green laps have no time (2022 R18, 2023 R3) · keep or drop deleted laps · used vs new tyres · does the rate differ by track and temperature (run Q10 on more races).


## Setup


In [ ]:
import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from f1_intelligence.config import load_ingestion_config

RAW = load_ingestion_config("../configs/ingestion.yaml").storage.raw_uri


def raw_table(table: str, session: str = "R") -> pd.DataFrame:
    """All ingested sessions of one raw table, as a single DataFrame."""
    return duckdb.sql(f"""
        SELECT *
        FROM read_parquet('{RAW}/{table}/**/*.parquet', hive_partitioning = true, union_by_name = true)
        WHERE Session = '{session}'
    """).df()


meta = duckdb.sql(f"""
    SELECT season AS Season, round AS Round, event_name, location, event_format, session_start_utc
    FROM read_json('{RAW}/metadata/**/*.json', hive_partitioning = false)
    WHERE session = 'R'
""").df()

laps = raw_table("laps").merge(meta, on=["Season", "Round"], how="left")
weather = raw_table("weather")
results = raw_table("results")
track_status = raw_table("track_status")
race_control = raw_table("race_control")

print(f"{meta.shape[0]} races | {len(laps):,} laps | seasons {sorted(meta.Season.unique())}")
{name: df.shape for name, df in dict(laps=laps, weather=weather, results=results,
                                      track_status=track_status, race_control=race_control).items()}

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 60)

# Raw time columns are integer nanoseconds; add a seconds version of each.
for col in ["Time", "LapTime", "LapStartTime", "PitInTime", "PitOutTime",
            "Sector1Time", "Sector2Time", "Sector3Time"]:
    laps[f"{col}_sec"] = laps[col] / 1e9
weather["Time_sec"] = weather["Time"] / 1e9
track_status["Time_sec"] = track_status["Time"] / 1e9

# Example race used throughout: Bahrain 2022 (season opener)
def one_race(df, season=2022, rnd=1):
    return df[(df["Season"] == season) & (df["Round"] == rnd)].copy()

## Q1 — What is one row?


In [ ]:
laps.head()

In [ ]:
laps.info()

In [ ]:
# Is (Season, Round, Driver, LapNumber) unique? → number of duplicate rows
laps.duplicated(subset=["Season", "Round", "Driver", "LapNumber"]).sum()

In [ ]:
# Race length per race (Round alone is ambiguous across seasons)
laps.groupby(["Season", "Round", "event_name"])["LapNumber"].max().head(10)

In [ ]:
# Laps completed per driver in one race: shows retirements
one_race(laps).groupby("Driver")["LapNumber"].max().sort_values().head(5)

> **Finding:** One row = one lap of one driver in one race. `(Season, Round, Driver, LapNumber)` has **0 duplicates**, so it is the key of the processed table.
> `Round` alone is ambiguous (round 6 is a different race each year), so always use `Season` + `Round`.
> Drivers with fewer laps than the race length retired (Bahrain 2022: GAS 45, VER 54 of 57).
> Times are stored as **integer nanoseconds** → convert to seconds.


## Q2 — Missing values


In [ ]:
laps.isna().sum()[lambda s: s > 0].sort_values(ascending=False)

In [ ]:
# Missing lap times per race
laps[laps["LapTime"].isna()].groupby(["Season", "Round", "event_name"]).size().sort_values(ascending=False).head(10)

In [ ]:
# Missing lap times by track status: is a safety car / red flag involved?
missing = laps[laps["LapTime"].isna()]
(pd.DataFrame({
    "safety_car_or_vsc": missing["TrackStatus"].str.contains("4|6"),
    "red_flag": missing["TrackStatus"].str.contains("5"),
}).value_counts())

> **Finding:** `LapStartDate` is always empty → drop it. Pit columns are empty except on pit laps, which is expected.
> `LapTime` is missing on **1,539 laps (1.5%)**, concentrated in a few races (Singapore 2022: 108).
> Of those, **1,252 are safety-car/VSC laps** and 131 red-flag laps; only 262 have neither.
> **Decision:** flag `has_lap_time`. These laps are excluded from degradation anyway.


## Q3 — Track status


In [ ]:
# Code book: one digit per status
track_status[["Status", "Message"]].drop_duplicates().sort_values("Status")

In [ ]:
# Lap time per TrackStatus combination (count = how many laps)
laps.groupby("TrackStatus")["LapTime_sec"].agg(["count", "mean"]).sort_values("count", ascending=False).head(12)

> **Finding:** Each digit is one status; a lap's string lists every status during that lap, in order (`41` = safety car, then all clear: the restart lap).
> Average lap: green **89.7 s**, safety car **122.8 s**, VSC **119.7 s**; red-flag laps can include the whole stoppage.
> **Decision:** don't use the ~40 combinations as categories. Create yes/no flags `yellow_in_lap`, `sc_in_lap`, `vsc_in_lap`, `red_in_lap`. For degradation, keep only fully green laps (`"1"`).


## Q4 — Pit laps


In [ ]:
# VER, Bahrain 2022: the laps either side of each pit stop
ver = one_race(laps).query("Driver == 'VER'")
is_pit_lap = ver["PitInTime"].notna() | ver["PitOutTime"].notna()
around_stop = is_pit_lap | is_pit_lap.shift(1, fill_value=False) | is_pit_lap.shift(-1, fill_value=False)
ver.loc[around_stop, ["LapNumber", "Stint", "Compound", "TyreLife", "LapTime_sec", "PitInTime_sec", "PitOutTime_sec"]]

> **Finding:** Normal lap ~99.5 s → **in-lap 102.1 s (+2.5)** → **out-lap 117.4 s (+18)** → first lap on new tyres **96.9 s**.
> The extra time on in/out laps is the **pit stop** (speed limit, stop, cold tyres), not tyre wear.
> The laps either side of the stop show what worn tyres cost: **2.6 s** (99.5 → 96.9).
> **Decision:** flag `is_pit_in_lap` / `is_pit_out_lap`; exclude them from degradation; use them later to estimate pit loss per track.


## Q5 — Outliers and odd laps


In [ ]:
print(laps["IsAccurate"].value_counts(), "\n")
print(laps["Deleted"].value_counts(), "\n")
laps["DeletedReason"].value_counts().head(5)

In [ ]:
three_drivers = one_race(laps).query("Driver in ['VER', 'LEC', 'HAM']")
sns.lineplot(data=three_drivers, x="LapNumber", y="LapTime_sec", hue="Driver")
plt.title("Raw lap times, Bahrain 2022: spikes are pit stops and safety car")
plt.ylabel("Lap time (s)")
plt.show()

> **Finding:** Raw lap times have big spikes: pit laps (+18 s) and safety-car laps (+30 s). Normal laps sit in a narrow band.
> `IsAccurate` is False on **13.6%** of laps, almost all pit laps, lap 1 and non-green laps, so it mostly repeats other flags.
> `Deleted` on **1.3%** of laps, almost all track limits (the car really drove the lap).
> **Decision:** keep both as flags. Open: whether to exclude deleted laps from degradation.


## Q6 — Tyres and stints


In [ ]:
laps["Compound"].value_counts(dropna=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.lineplot(data=three_drivers, x="LapNumber", y="Stint", hue="Driver", ax=axes[0])
axes[0].set_title("Stint: steps up at each pit stop")
sns.lineplot(data=three_drivers, x="LapNumber", y="TyreLife", hue="Driver", ax=axes[1])
axes[1].set_title("TyreLife: resets at each pit stop")
plt.show()

In [ ]:
# Tyre age at the first lap of each stint: new vs used sets
stint_first = laps.sort_values("LapNumber").groupby(["Season", "Round", "Driver", "Stint"]).first()
stint_first.groupby("FreshTyre")["TyreLife"].describe()[["count", "min", "50%", "max"]]

In [ ]:
# How long are stints? (laps per stint)
laps.groupby(["Season", "Round", "Driver", "Stint"]).size().describe()

> **Finding:** Dry compounds dominate; **INTERMEDIATE (5,945) and WET (461)** laps are rain laps.
> `"None"` (426) and `"nan"` (354) are **text, not real missing values** → convert to missing in the processed layer.
> `Stint` is a counter of tyre sets (staircase); `TyreLife` is the age of the current set (sawtooth, resets at each stop).
> New sets start at tyre age 1–2; **used sets start older** (median 4).
> Stints are typically **19 laps** (middle half: 11–27); some are only 1–4 laps.
> **Decision:** `is_wet_compound` flag; for slopes, skip stints shorter than 5 laps.


## Q7 — Weather


In [ ]:
weather[["Season", "Round", "Time_sec", "AirTemp", "TrackTemp", "Rainfall"]].head(5)

In [ ]:
# Time between weather readings (seconds)
weather.groupby(["Season", "Round"])["Time_sec"].diff().describe()

In [ ]:
bahrain_weather = one_race(weather)
print("Within Bahrain 2022:\n", bahrain_weather["TrackTemp"].describe()[["mean", "min", "max"]], "\n")
print("Across races (mean track temp per race):\n", weather.groupby(["Season", "Round"])["TrackTemp"].mean().describe()[["mean", "min", "max"]])
bahrain_weather.plot(x="Time_sec", y="TrackTemp", title="Track temperature during Bahrain 2022")
plt.show()

In [ ]:
# Races with rain: how many rain readings, and how many laps on wet tyres?
rain = weather.groupby(["Season", "Round"])["Rainfall"].sum().rename("rain_readings")
wet_laps = laps[laps["Compound"].isin(["INTERMEDIATE", "WET"])].groupby(["Season", "Round"]).size().rename("wet_tyre_laps")
pd.concat([rain, wet_laps], axis=1).fillna(0).astype(int).query("rain_readings > 0 or wet_tyre_laps > 0").sort_values("wet_tyre_laps", ascending=False).head(12)

> **Finding:** One reading every **~60 s** (median 60.0 s).
> Track temperature changes a little **within** a race (Bahrain 2022: 26–32 °C, cooling as the evening goes on) and a lot **between** races (17–54 °C).
> The rain sensor and the tyres disagree: Imola 2022 had **0 rain readings but 329 laps on wet tyres** (rain before the start), while Japan 2022 had many rain readings.
> **Decision:** attach to each lap the latest weather reading before it started; `track_temp` is a model feature; mark wet laps by **compound**, not by `Rainfall`.


## Q8 — Fuel effect


In [ ]:
# Clean lap = a normal racing lap: only fuel and tyres should explain its time
laps["is_clean"] = (
    (laps["LapNumber"] > 1)
    & laps["LapTime_sec"].notna()
    & laps["PitInTime"].isna()
    & laps["PitOutTime"].isna()
    & (laps["TrackStatus"] == "1")
    & laps["Compound"].isin(["SOFT", "MEDIUM", "HARD"])
)
laps["is_clean"].mean()  # share of laps that are clean

In [ ]:
ver_clean = one_race(laps).query("Driver == 'VER' & is_clean")
sns.lineplot(data=ver_clean, x="LapNumber", y="LapTime_sec", hue="Stint", style="Compound", markers=True)
plt.title("VER clean laps, Bahrain 2022: up within each stint, down across stints")
plt.show()

In [ ]:
# Average of the first 3 clean laps of each stint
ver_clean.sort_values("LapNumber").groupby(["Stint", "Compound"]).head(3).groupby(["Stint", "Compound"])["LapTime_sec"].mean()

> **Finding:** **79%** of laps are clean (no lap 1, pit laps, non-green laps or wet tyres).
> For VER: lap time **rises within each stint** (tyres wearing) but each stint **starts faster** than the previous one: 98.27 → 97.40 → 96.42 → 95.55 s.
> That drop is mostly **fuel**: there is no refuelling, so the car gets lighter every lap. About **0.06 s/lap**.
> Across all races (regression with tyre age and driver): median **0.056 s/lap**, from ~0.03 (Monaco) to ~0.08 (Spa); it also includes the track getting faster.
> **Why it matters:** within a stint, fuel makes the car faster while tyres make it slower, so raw lap times **understate** degradation.
> **Decision:** fuel-correct before measuring degradation: `lap_time + fuel_effect × LapNumber`, with `fuel_effect` per track in config.


## Q9 — Traffic (dirty air)


In [ ]:
# Gap to the car ahead: seconds after the car one position ahead crossed the line, same lap
laps = laps.sort_values(["Season", "Round", "LapNumber", "Position"])
laps["gap_ahead_sec"] = laps.groupby(["Season", "Round", "LapNumber"])["Time_sec"].diff()
print("negative gaps (invalid):", (laps["gap_ahead_sec"] < 0).sum())
laps.loc[laps["gap_ahead_sec"] < 0, "gap_ahead_sec"] = np.nan

one_race(laps).query("LapNumber == 5")[["Position", "Driver", "Time_sec", "gap_ahead_sec"]]

In [ ]:
# Ocon, Bahrain 2022: catching a car (gap shrinks) → slower; free air again → faster
one_race(laps).query("Driver == 'OCO' & is_clean & 32 <= LapNumber <= 39")[["LapNumber", "TyreLife", "gap_ahead_sec", "LapTime_sec"]]

In [ ]:
# All races: clean laps, lap time relative to the driver's own median in that race
clean = laps[laps["is_clean"]].copy()
clean["rel_sec"] = clean["LapTime_sec"] - clean.groupby(["Season", "Round", "Driver"])["LapTime_sec"].transform("median")
clean["gap_bin"] = pd.cut(clean["gap_ahead_sec"], [0, 0.5, 1, 1.5, 2, 3, 5, 10, 1000])
clean.groupby("gap_bin", observed=True)["rel_sec"].agg(["count", "median"])

In [ ]:
laps["in_traffic"] = laps["gap_ahead_sec"] < 1.0   # leader (NaN gap) counts as free air
laps["in_traffic"].mean()

> **Finding:** Gap ahead = seconds after the car one position ahead crossed the line on the same lap. **59 negative gaps** (impossible) are set to missing.
> Ocon shows the effect: as he catches a car (gap 3.9 → 0.5 s) his laps slow (98.1 → 98.9 s); in free air again, he is 0.6 s faster on an older tyre.
> Across all races, laps within 0.5 s look ~0.4 s slower than the driver's median, but that **overstates** it: close gaps happen early in the race, when cars are heavy. The real effect is roughly **0.1–0.5 s within ~1 s**.
> **Decision:** store `gap_ahead_sec`; flag `in_traffic` = gap < 1 s (24% of laps); exclude from degradation. The leader (no car ahead) counts as free air.


## Q10 — Degradation: lap time vs tyre age (Bahrain 2022)


In [ ]:
FUEL_EFFECT = 0.06   # s/lap; median across races (varies by track: ~0.03 Monaco to ~0.08 Spa)

race = one_race(laps).query("is_clean & ~in_traffic & FreshTyre").copy()
race["fuel_corrected_sec"] = race["LapTime_sec"] + FUEL_EFFECT * race["LapNumber"]

race = race.sort_values(["Driver", "Stint", "LapNumber"])
race["stint_start"] = race.groupby(["Driver", "Stint"])["fuel_corrected_sec"].transform(lambda s: s.head(3).mean())
race["delta"] = race["fuel_corrected_sec"] - race["stint_start"]
race[["Driver", "Stint", "Compound", "LapNumber", "TyreLife", "fuel_corrected_sec", "stint_start", "delta"]].head(8)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.lineplot(data=race, x="TyreLife", y="fuel_corrected_sec", hue="Compound", style="Compound", markers=True, ax=axes[0])
axes[0].set_title("Fuel-corrected lap time vs tyre age")
sns.lineplot(data=race, x="TyreLife", y="delta", hue="Compound", style="Compound", markers=True, ax=axes[1])
axes[1].set_title("Delta vs own stint start = degradation curve")
plt.show()

In [ ]:
def stint_slope(stint):
    """Degradation rate of one stint: seconds lost per lap of tyre age."""
    if len(stint) < 5:  # too few laps to trust a line
        return np.nan
    slope, intercept = np.polyfit(stint["TyreLife"], stint["delta"], 1)
    return slope

slopes = (race.groupby(["Driver", "Stint", "Compound"])
              .apply(stint_slope, include_groups=False)
              .rename("slope").reset_index())
slopes.groupby("Compound")["slope"].agg(["count", "median", "mean"])

In [ ]:
# Sensitivity: do the conclusions hold for a different fuel effect?
def median_slopes(fuel_effect):
    r = race.copy()
    r["fuel_corrected_sec"] = r["LapTime_sec"] + fuel_effect * r["LapNumber"]
    r["delta"] = r["fuel_corrected_sec"] - r.groupby(["Driver", "Stint"])["fuel_corrected_sec"].transform(lambda s: s.head(3).mean())
    return r.groupby(["Driver", "Stint", "Compound"]).apply(stint_slope, include_groups=False).groupby("Compound").median()

pd.DataFrame({f"fuel {f}": median_slopes(f) for f in [0.04, 0.06, 0.075]})

> **Finding:** Using clean, traffic-free laps on new tyres, fuel-corrected, and each stint compared with its own first 3 laps:
> the delta rises **almost linearly** with tyre age (~2 s after 15 laps).
> Median degradation per stint: **SOFT 0.18 · MEDIUM 0.15 · HARD 0.13 s/lap**: softer tyres wear faster, as expected. HARD is based on only 5 stints.
> **Sensitivity:** within a stint, `TyreLife` and `LapNumber` both go up by 1 each lap, so changing the fuel value by 0.015 shifts **every slope by exactly 0.015**. The order of the compounds stays the same, but the absolute rate depends on getting the fuel effect right.
> **Decision:** the Stage 1 base model can start **linear in tyre age**, per compound. Next: repeat on other tracks to see how much the rate depends on track and temperature.


## Notes / open questions
